# 🌿 Rainforest Dashboard — Data Acquisition
Pulls annual PRODES deforestation data from TerraBrasilis and loads it into the Supabase `Rainforest` schema.

**Run once** to populate the database. Re-run to refresh (truncates and reloads).

## Prerequisites
- Set environment variable `SUPABASE_DB_PASSWORD` in Deepnote project settings
- Supabase schema `Rainforest` must exist (run `db/rainforest_schema.sql` first)

In [ ]:
import requests
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
import os

# Supabase direct DB connection (not PostgREST — needed for INSERT)
SUPABASE_HOST = "supabase.butscher.cloud"
SUPABASE_DB   = "postgres"
SUPABASE_USER = "postgres"
SUPABASE_PASS = os.environ["SUPABASE_DB_PASSWORD"]
SUPABASE_PORT = 5432

print("✓ Config loaded")

## Step 1 — Fetch PRODES data from TerraBrasilis WFS

In [ ]:
# TerraBrasilis WFS endpoint for annual PRODES deforestation by state/biome
WFS_URL = (
    "http://terrabrasilis.dpi.inpe.br/geoserver/prodes-amazon-nb/ows"
    "?service=WFS&version=1.0.0&request=GetFeature"
    "&typeName=prodes-amazon-nb:yearly_deforestation_biome"
    "&outputFormat=application/json"
)

print("Fetching from TerraBrasilis...")
resp = requests.get(WFS_URL, timeout=120)
resp.raise_for_status()

features = resp.json()["features"]
rows = [f["properties"] for f in features]
df_raw = pd.DataFrame(rows)

print(f"✓ {len(df_raw)} rows fetched")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head(3)

## Step 2 — Transform
Inspect column names above and adjust the rename mapping if needed.

In [ ]:
# --- Adjust column names after inspecting output of Cell 4 ---
# Common TerraBrasilis column names (verify and update as needed):
# 'ano' or 'year', 'state' or 'uf', 'classname' or 'class_name'

# Attempt auto-detection
col_map = {}
for col in df_raw.columns:
    c = col.lower()
    if c in ("ano", "year"):
        col_map[col] = "year"
    elif c in ("state", "uf", "estado", "state_name"):
        col_map[col] = "state_name"
    elif c in ("classname", "class_name", "class", "tipo", "type"):
        col_map[col] = "class_name"
    elif c in ("area", "areakm", "area_km", "desmatamento"):
        col_map[col] = "area_km2"

print(f"Detected mapping: {col_map}")

df = df_raw.rename(columns=col_map).copy()
df = df[["year", "state_name", "class_name", "area_km2"]].dropna()
df["year"]     = df["year"].astype(int)
df["area_km2"] = df["area_km2"].astype(float).round(2)

print(f"✓ {len(df)} rows after transform, years {df.year.min()}–{df.year.max()}")
df.head(3)

## Step 3 — Calculate cumulative area

In [ ]:
df = df.sort_values(["state_name", "class_name", "year"]).reset_index(drop=True)
df["accumulated_km2"] = (
    df.groupby(["state_name", "class_name"])["area_km2"]
    .cumsum()
    .round(2)
)
print(f"✓ Cumulative area calculated")
df.head(5)

## Step 4 — Load into Supabase

In [ ]:
conn = psycopg2.connect(
    host=SUPABASE_HOST, dbname=SUPABASE_DB,
    user=SUPABASE_USER, password=SUPABASE_PASS, port=SUPABASE_PORT
)
cur = conn.cursor()

# Truncate existing data for clean reload
cur.execute('TRUNCATE "Rainforest".fact_deforestation CASCADE')
cur.execute('TRUNCATE "Rainforest".dim_state CASCADE')
cur.execute('TRUNCATE "Rainforest".dim_class CASCADE')

# Insert dim_state
state_map = {}
for s in df["state_name"].unique():
    cur.execute(
        'INSERT INTO "Rainforest".dim_state (state_code, state_name) '
        'VALUES (%s, %s) RETURNING state_id',
        (s[:2].upper(), s)
    )
    state_map[s] = cur.fetchone()[0]

# Insert dim_class
class_map = {}
for c in df["class_name"].unique():
    cur.execute(
        'INSERT INTO "Rainforest".dim_class (class_name) '
        'VALUES (%s) RETURNING class_id',
        (c,)
    )
    class_map[c] = cur.fetchone()[0]

# Insert fact_deforestation
records = [
    (int(r.year), state_map[r.state_name], class_map[r.class_name],
     float(r.area_km2), float(r.accumulated_km2))
    for r in df.itertuples()
]
execute_values(
    cur,
    'INSERT INTO "Rainforest".fact_deforestation '
    '(year, state_id, class_id, area_km2, accumulated_km2) VALUES %s',
    records
)
conn.commit()
conn.close()

print(f"✓ {len(state_map)} states, {len(class_map)} classes, {len(records)} fact rows loaded")

## ✅ Done
Data is now in Supabase. Verify in the Supabase SQL Editor:
```sql
SELECT year, COUNT(*) as rows FROM "Rainforest".fact_deforestation GROUP BY year ORDER BY year;
```